# Preprocessing

Removes the students who did not consent and joins the sentences of each message into one row.

* **Input:** `source/conversations.csv`, `source/consent.csv`, `source/activities.csv`
* **Output:** `df_chats.csv`
* **Check:** 159 conversations from 78 students (23, 35, 52 and 49 per activity)


In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

# Dataset to work on: 'data' is the study data, 'data_example' the synthetic example.
DATA = Path('..') / os.environ.get('DATASET_FOLDER', 'data')

## Read the conversations, the consent and the activities

In [2]:
# Read consent
consent_df = pd.read_csv(DATA / 'source' / 'consent.csv')

# Read the activities (name and week of each one)
activities_df = pd.read_csv(DATA / 'source' / 'activities.csv')

# Read the data
excel_data_df = pd.read_csv(DATA / 'source' / 'conversations.csv')

# Merge with the consent data so that only the students who consented are used
df = excel_data_df.merge(consent_df, on='user_id', how='left')

df.head()

,activity_id,user_id,role,message_num,sentence_num,sentence_orig,sentence_en,consent_use,consent_grade,consent_share
0,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,model,0,0,"Bonjour ! 😸 Je suis SIMBA, et je vais vous aid...","Hello! 😸 I am SIMBA, and I will help you think...",YES,YES,YES
1,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,model,0,1,"D'après vous, quelles sont les 2 à 5 idées pri...","In your opinion, what are the 2 to 5 main idea...",YES,YES,YES
2,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,model,0,2,Quels sont les points du cours qui vous semble...,What are the points of the course that still s...,YES,YES,YES
3,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,model,0,3,"Allez-y, partagez vos réflexions sur la premiè...","Go ahead, share your thoughts on the first que...",YES,YES,YES
4,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,user,1,0,les idées principales sont : les diagrammes T-...,"The main ideas are: T-V diagrams, P-V diagrams...",YES,YES,YES


In the original data there was one inconsistency between the two sources of consent (the
`StudentsIdConsent` sheet and the `synthesis` file): student 35 does not consent to the use of their
data in the first one and does consent in the second. The consent published here is the one of the
`synthesis` file, which is what the analysis used.

In [3]:
# Remove the users who did not consent to the use of their data
df = df[df['consent_use'] == 'YES']
df = df[df['consent_grade'] == 'YES']
df = df[df['consent_share'] == 'YES']

df.drop(columns=['consent_use', 'consent_share', 'consent_grade'], inplace=True)

df = df[df['sentence_en'].notna()]

print(len(df), 'sentences,', df['user_id'].nunique(), 'students')
df.head()

4171 sentences, 78 students


,activity_id,user_id,role,message_num,sentence_num,sentence_orig,sentence_en
0,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,model,0,0,"Bonjour ! 😸 Je suis SIMBA, et je vais vous aid...","Hello! 😸 I am SIMBA, and I will help you think..."
1,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,model,0,1,"D'après vous, quelles sont les 2 à 5 idées pri...","In your opinion, what are the 2 to 5 main idea..."
2,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,model,0,2,Quels sont les points du cours qui vous semble...,What are the points of the course that still s...
3,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,model,0,3,"Allez-y, partagez vos réflexions sur la premiè...","Go ahead, share your thoughts on the first que..."
4,asst_Get3WE5ozTjkHcdGhnqGGqgL,s140,user,1,0,les idées principales sont : les diagrammes T-...,"The main ideas are: T-V diagrams, P-V diagrams..."


## Sort the conversations

**Change with respect to the original notebook.** The order of the rows was the one of the source
file, which was not the same for every coding scheme. Sorting the conversations by activity (week) and
student fixes the order of the cases in the event log, and with it the result of the clustering in the
notebook `06_clustering` (see the note there).

In [4]:
week = activities_df.set_index('activity_id')['week']

df['week'] = df['activity_id'].map(week)
df = df.sort_values(['week', 'user_id', 'message_num', 'sentence_num'], kind='stable')

## Join utterances of the same agent as one

In [5]:
df['same_role'] = np.equal(df['role'], df['role'].shift(+1))
df['same_convo'] = np.equal(df['activity_id'] + df['user_id'],
                            (df['activity_id'] + df['user_id']).shift(+1))
# Work out when to consolidate the activities into one and when not to
df['consolidar'] = np.logical_and(df['same_role'], df['same_convo'])

count = 0
df['group'] = 0
aux = list(df.columns)
idx_consolidar = aux.index('consolidar')
idx_group = aux.index('group')
for i in range(len(df)):
    if df.iloc[i, idx_consolidar] == False:
        count += 1
    df.iloc[i, idx_group] = count

# Create the new dataframe with the consolidated activities

df['sentence_en'] = df['sentence_en'].astype(str)

# Keep only the relevant columns
df_consolidate = df.groupby(['group']).agg({
    'activity_id': 'first',
    'user_id': 'first',
    'group': 'first',
    'role': 'first',
    'message_num': 'first',
    'sentence_num': 'first',
    'sentence_en': '\n'.join,  # Concatenate activities
  }).reset_index(drop=True)

df_consolidate.head(10)

,activity_id,user_id,group,role,message_num,sentence_num,sentence_en
0,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,1,model,0,0,"Hello! 😸 I am SIMBA, and I will help you think..."
1,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,2,user,1,0,Can you clarify the difference between an isol...
2,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,3,model,2,0,Sure! 🌟\n1. **Isolated system**: It cannot exc...
3,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,4,user,3,0,Can you make a big summary of the important no...
4,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,5,model,4,0,Sure! Here is a big summary of the important n...
5,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,6,user,5,0,Can you develop the different types of transfo...
6,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,7,model,6,0,Sure! Here is a development on the different t...
7,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s109,8,model,0,0,"Hello! 😸 I am SIMBA, and I will help you think..."
8,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s109,9,user,1,0,The first idea is that P and V are not necessa...
9,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s109,10,model,2,0,"That's a good observation! 😊 Indeed, pressure ..."


In [6]:
len(df_consolidate['user_id'].unique())

78

# Saving output

In [7]:
# Relative path
path = DATA

# Create the folder if it does not exist
path.mkdir(parents=True, exist_ok=True)

out_path = path / 'df_chats.csv'

df_consolidate.to_csv(out_path, index=False)

## Description of the corpus

Added for the publication: the counts reported in the paper, so that the dataset can be checked at a
glance.

In [8]:
conversations = df_consolidate[['activity_id', 'user_id']].drop_duplicates()

print('conversations:', len(conversations))
print('students:', conversations['user_id'].nunique())

participation = conversations.groupby('activity_id').size().rename('n_conversations')
activities_df.set_index('activity_id').join(participation).sort_values('week')

conversations: 159
students: 78


,activity_name,week,graded,n_conversations
activity_id,,,,
asst_ncmAPAoB3MT4TIBrhhRAjpJ9,Summary 2,2,False,23
asst_ZZ2f7t77l5PKcfG6pHkVMQDz,Summary 3,3,False,35
asst_xRGU7Xk0G1eHkruFzY1kpaTO,Summary 4,4,True,49
asst_Get3WE5ozTjkHcdGhnqGGqgL,Summary 5,5,True,52


In [10]:
per_student = conversations['user_id'].value_counts()

print('conversations per student: min', per_student.min(),
      '| median', int(per_student.median()), '| max', per_student.max())

per_student.value_counts().sort_index().rename_axis('conversations').rename('n_students').to_frame()

conversations per student: min 1 | median 2 | max 3


,n_students
conversations,
1,27
2,21
3,30
